# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a transparent Week-4 baseline from observable, pre-decision signals only. The rule is deliberately simple and frozen before Week-5 modeling.

**Lane:** content refresh / search visibility

**Decision slice:** bundled anonymized starter dataset. The starter slice is a trailing-90-day snapshot; it contains no FlyRank product flags, so this audit validates observable proxies rather than reverse-engineering a hidden product score.

## 1. Signal checks and rule reasoning

### Signal 1 — staleness
**Hypothesis:** pages that have not been updated for a long time are more plausible refresh candidates. This is the observable signal behind the session's refresh/staleness flag logic. I will bucket `days_since_last_update` and compare each bucket's declining rate, using `n` in every bucket. A positive step-up across older buckets supports the hypothesis; a flat or reversed pattern is evidence against treating staleness as strong by itself.

**Verdict:** CONFIRMED / OPPOSITE / MIXED / FALSE (filled by the executed cell below).

### Signal 2 — search visibility / volume
**Hypothesis:** a refresh is more actionable when the page has meaningful search exposure. This is the observable signal behind the session's quick-win / volume logic. I will bucket `impressions_90d` and compare the same outcome rate, again printing `n`. Because low-volume position and CTR are noisy, volume is used as an opportunity gate rather than as proof of decline.

**Verdict:** CONFIRMED / OPPOSITE / MIXED / FALSE (filled by the executed cell below).

### Rule in plain words
A page rises to the top of the baseline queue when it is **old since its last update** and has **enough search visibility to make a refresh worth reviewing**. Among pages meeting those conditions, more visible pages rank first. The rule does not use `trend_pct`, `trend_direction`, `is_declining_label`, or any future-window field.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks' and ROOT.parent.name == 'work':
    ROOT = ROOT.parent.parent
CSV_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
OUT_DIR = ROOT / 'work' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
required = ['content_id','client_id','days_since_last_update','impressions_90d','avg_position','ctr','content_age_days','word_count']
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Safe numeric coercion. IDs stay as context only.
for c in required[2:]:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# The data dictionary says avg_position=0 means no position data. Keep those rows out of
# CTR/position interpretation, but they may still participate in the refresh rule.

# Outcome proxy used ONLY to audit the signal in this retrospective teaching slice.
# It is never used to score/rank rows.
if 'is_declining_label' in df.columns:
    audit_y = pd.to_numeric(df['is_declining_label'], errors='coerce')
elif 'trend_direction' in df.columns:
    audit_y = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
else:
    raise ValueError('Need the existing audit outcome to compute the signal tables.')

# Staleness buckets use explicit, human-readable thresholds.
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=['0-30','31-90','91-180','181-365','365+'],
    right=True,
)
stale_tab = (
    pd.DataFrame({'bucket': df['staleness_bucket'], 'y': audit_y})
    .dropna(subset=['bucket'])
    .groupby('bucket', observed=False)['y']
    .agg(n='size', declining_rate='mean')
    .reset_index()
)
print('SIGNAL 1 — STALENESS BUCKET TABLE')
display(stale_tab.style.format({'declining_rate':'{:.3f}'}))

# Volume buckets mirror the data dictionary's transparency tiers.
df['volume_bucket'] = pd.cut(
    df['impressions_90d'],
    bins=[-np.inf, 99, 299, 2999, 29999, np.inf],
    labels=['<100','100-299','300-2999','3000-29999','30000+'],
    right=True,
)
volume_tab = (
    pd.DataFrame({'bucket': df['volume_bucket'], 'y': audit_y})
    .dropna(subset=['bucket'])
    .groupby('bucket', observed=False)['y']
    .agg(n='size', declining_rate='mean')
    .reset_index()
)
print('SIGNAL 2 — SEARCH VISIBILITY / VOLUME BUCKET TABLE')
display(volume_tab.style.format({'declining_rate':'{:.3f}'}))

# Deterministic verdict helper. We judge direction by endpoint movement and monotonic fit.
def verdict(tab: pd.DataFrame) -> str:
    rates = tab['declining_rate'].to_numpy(dtype=float)
    if len(rates) < 3 or np.isnan(rates).all():
        return 'FALSE'
    diffs = np.diff(rates)
    if np.all(diffs >= -0.01) and rates[-1] - rates[0] >= 0.05:
        return 'CONFIRMED'
    if np.all(diffs <= 0.01) and rates[0] - rates[-1] >= 0.05:
        return 'OPPOSITE'
    return 'MIXED'

print(f'Verdict — staleness: {verdict(stale_tab)}')
print(f'Verdict — volume: {verdict(volume_tab)}')

### Rule design choice

The scoring rule below is intentionally not fitted. It uses two simple gates: **stale** = `days_since_last_update >= 180`, and **visible** = `impressions_90d >= 500`. The score is `stale * visible * log1p(impressions_90d)`. This preserves the session's idea: a rule should be human-readable, threshold-based, and easy to challenge.

The rule emits exactly **one reason code**: `stale_visible_page` when both gates fire, otherwise `general_review`. The action label is `refresh` for the flagged group and `monitor` otherwise.

## 2. Build the ranked queue (writes the CSV)

The queue is written to `work/outputs/baseline_action_score.csv`. The CSV is intentionally ignored by git because the repo's leak-guard says to regenerate it from the notebook.

In [ ]:
# Transparent baseline: no fitted weights, no trend/label features, no future window.
work = df.copy()
work['stale'] = (work['days_since_last_update'] >= 180).astype(int)
work['visible'] = (work['impressions_90d'] >= 500).astype(int)
work['score'] = work['stale'] * work['visible'] * np.log1p(work['impressions_90d'].clip(lower=0))
work['reason_code'] = np.where(work['stale'].eq(1) & work['visible'].eq(1), 'stale_visible_page', 'general_review')
work['action'] = np.where(work['reason_code'].eq('stale_visible_page'), 'refresh', 'monitor')
work = work.sort_values(['score','impressions_90d','days_since_last_update'], ascending=[False,False,False]).reset_index(drop=True)
work['rank'] = np.arange(1, len(work) + 1)

# Precision@K is diagnostic only; it uses the retrospective label and never enters score creation.
work['audit_label'] = audit_y.reset_index(drop=True)

def precision_at_k(frame: pd.DataFrame, k: int) -> float:
    return float(frame.head(min(k, len(frame)))['audit_label'].mean()) if len(frame) else 0.0

p10 = precision_at_k(work, 10)
p50 = precision_at_k(work, 50)
base_rate = float(work['audit_label'].mean())
print(f'Base rate: {base_rate:.3f}')
print(f'Precision@10: {p10:.3f}')
print(f'Precision@50: {p50:.3f}')

queue_cols = ['rank','content_id','client_id','score','reason_code','action','impressions_90d','days_since_last_update','avg_position','ctr','content_age_days','word_count']
queue = work[queue_cols].copy()
queue.to_csv(OUT_DIR / 'baseline_action_score.csv', index=False)

# Save a small receipt JSON; CSV remains intentionally ignored by git.
receipt = {
    'rows': int(len(queue)),
    'base_rate_declining': base_rate,
    'precision_at_10': p10,
    'precision_at_50': p50,
    'rule': 'stale=(days_since_last_update>=180); visible=(impressions_90d>=500); score=stale*visible*log1p(impressions_90d)',
    'reason_codes': ['stale_visible_page', 'general_review'],
    'action_labels': ['refresh', 'monitor'],
    'leakage_note': 'trend_pct, trend_direction, is_declining_label and future-window fields were not used in score construction.'
}
(OUT_DIR / 'w04_baseline_receipt.json').write_text(json.dumps(receipt, indent=2))

display(queue.head(10))
print(f'Wrote {len(queue):,} rows to {OUT_DIR / "baseline_action_score.csv"}')

## 3. Top-20 skeptic review

The assignment asks for the top ten; the skeleton asks for top twenty. I review twenty because it makes weak picks easier to spot. Each line names the action, why it ranked, and a concrete condition that would make the recommendation wrong.

In [ ]:
top20 = work.head(20).copy()
review_rows = []
for _, r in top20.iterrows():
    if r['reason_code'] == 'stale_visible_page':
        why = f"old update ({int(r['days_since_last_update'])}d) + visible ({int(r['impressions_90d']):,} impressions)"
        wrong = 'wrong if the page is intentionally evergreen, already scheduled for refresh, or the traffic is low-quality/unconvertible.'
        action = 'refresh'
    else:
        why = 'does not meet both baseline gates; kept only as a low-priority comparator'
        wrong = 'wrong if manual review reveals a hidden business priority that the observable search metrics miss.'
        action = 'monitor'
    review_rows.append({
        'rank': int(r['rank']),
        'action': action,
        'why_it_is_here': why,
        'what_would_make_it_wrong': wrong
    })
review = pd.DataFrame(review_rows)
display(review)

## 4. Weak picks + leakage check

The most suspicious baseline picks are pages with high impressions but no compelling editorial opportunity beyond staleness. The rule is therefore a review queue, not an automatic publish/replace command.

**Leakage check:** the score uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_pct`, `trend_direction`, `is_declining_label`, model outputs, product flags, URLs, titles, keywords, or any forward window. `avg_position` and `ctr` are retained in the exported queue only for human review; they are not score inputs.

In [ ]:
score_inputs = ['days_since_last_update','impressions_90d']
forbidden_inputs = ['trend_pct','trend_direction','is_declining_label','future','next','label']
print('Score inputs:', score_inputs)
print('Forbidden/label-derived inputs are not referenced in the score expression.')
print('Weak-pick note: inspect the lowest-confidence rows manually before acting; stale + visible can still be the wrong editorial choice.')
print(f'Top-10 review rows available: {min(10, len(work))}')
print('Queue file exists:', (OUT_DIR / 'baseline_action_score.csv').exists())
assert not any(x in "days_since_last_update impressions_90d" for x in forbidden_inputs)

## Self-check

- [x] Two visible bucket tables with `n`; at least one signal is flag-linked (staleness / refresh logic).
- [x] One transparent rule with a score, exactly one reason code per row, and an action label.
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 skeptic review includes action, why, and what would make it wrong.
- [x] No label-derived or future-window fields are used as score inputs.
- [x] CSV is left uncommitted by design; the JSON receipt is a small reproducible run artifact.

**Important:** run this notebook top-to-bottom in Colab or locally, save the executed `.ipynb`, then commit it to the repo.